# 实验
**注意此实验全程使用Qwen模型**

## 前提

In [1]:
# 导入 所需 库

import os
#换用中国镜像站
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
#关闭SSL
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["NO_PROXY"] = "hf-mirror.com,cdn-lfs.hf-mirror.com"
for key in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    os.environ.pop(key, None)


import math
import copy
import warnings
from typing import Optional, Dict, List, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

import pandas as pd
import numpy as np
from tqdm import tqdm

warnings.filterwarnings('ignore')
from aeloru_layer import AeloruConfig , AeloruLayer , inject_aeloru , test_aeloru

import urllib3
urllib3.disable_warnings()

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

os.environ["PYTHONIOENCODING"] = "utf-8"

e:\Conda\envs\Qelys\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 可选 检验Aeloru 是否正常

test_aeloru()


In [ ]:
#可选 注入Aeloru 到模型中 是否正常

# 1. 定义一个极简的“模型” (例如一个简单的MLP)
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)
print("原始模型结构:", model)

# 2. 配置 Aeloru
# 只注入 Linear 层，秩为 4
cfg = AeloruConfig(r=4, lora_alpha=8.0, use_relora=False) 

# 3. 执行注入 (实验核心)
# 这里会将 Sequential 中的所有 Linear 替换为 AeloruLayer
inject_aeloru(model, target_names=["0", "2"], cfg=cfg) 
# 注：这里的 target_names=["0", "2"] 是因为 Sequential 的子模块索引是字符串

print("\n注入后模型结构:", model)

# 4. 验证：进行一次前向传播
model.eval()
x = torch.randn(1, 10) # 模拟输入

with torch.no_grad():
    output = model(x)

print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"输出值: {output.item():.4f}")
print("✅ 简易注入实验成功！模型能正常运行。")


## 单任务性能对比实验（证明 "不牺牲单任务换连续学习"）
> 实验目的：证明 Aeloru 在单任务微调上，性能和主流 PEFT 方法相当，没有为了连续学习牺牲基础能力
### 实验设计：
1. 分别用 LoRA、DoRA、ReLoRA、Aeloru 在上述 5 个数据集上做单任务微调
2. 记录每个方法在每个数据集上的最终准确率和困惑度
3. 用表格呈现结果，最后一行加平均性能

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = "../../models/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 128
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 2e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
SEED = 42

AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_AMP = True

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_CONFIGS = [
    {"name": "glue", "config": "sst2", "text_cols": ["sentence"], "label_col": "label", "num_labels": 2},
    {"name": "glue", "config": "mrpc", "text_cols": ["sentence1", "sentence2"], "label_col": "label", "num_labels": 2},
    {"name": "glue", "config": "qnli", "text_cols": ["question", "sentence"], "label_col": "label", "num_labels": 2},
    {"name": "glue", "config": "rte",  "text_cols": ["sentence1", "sentence2"], "label_col": "label", "num_labels": 2},
    {"name": "glue", "config": "cola", "text_cols": ["sentence"], "label_col": "label", "num_labels": 2},
]

LORA_R = 8
LORA_ALPHA = 4.0
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj"]


# ========================= ReLoRA =========================

class ReLoRATrainer:
    def __init__(self, model, merge_every: int = 1000, lr: float = LR):
        self.model = model
        self.merge_every = merge_every
        self.lr = lr

    def maybe_merge_and_reinit(self, global_step: int):
        if global_step > 0 and global_step % self.merge_every == 0:
            print(f"[ReLoRA] Merging at step {global_step}...")
            self._merge_lora_weights()
            self._reset_lora_parameters()

    def _merge_lora_weights(self):
        for module in self.model.named_modules():
            if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
                scaling = getattr(module, 'scaling', module.lora_alpha / module.r)
                delta = (module.lora_B @ module.lora_A) * scaling
                module.base_layer.weight.data.add_(delta.t())

    def _reset_lora_parameters(self):
        for module in self.model.named_modules():
            if hasattr(module, 'lora_A') and hasattr(module, 'lora_B'):
                nn.init.kaiming_uniform_(module.lora_A, a=math.sqrt(5))
                nn.init.zeros_(module.lora_B)


# ========================= 数据集 =========================

def load_and_preprocess_dataset(ds_cfg: Dict, tokenizer):
    raw = load_dataset(ds_cfg["name"], ds_cfg["config"])
    text_cols = ds_cfg["text_cols"]
    label_col = ds_cfg["label_col"]

    def tokenize_fn(examples):
        if len(text_cols) == 1:
            return tokenizer(examples[text_cols[0]], truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)
        else:
            return tokenizer(examples[text_cols[0]], examples[text_cols[1]],
                           truncation=True, max_length=MAX_SEQ_LENGTH, padding=False)

    encoded = raw.map(tokenize_fn, batched=True)
    cols_to_remove = list(text_cols)
    for split in encoded.keys():
        for col in list(encoded[split].column_names):
            if col not in ["input_ids", "attention_mask", label_col]:
                cols_to_remove.append(col)
    cols_to_remove = list(set(cols_to_remove))
    encoded = encoded.remove_columns(cols_to_remove)
    if label_col != "labels":
        encoded = encoded.rename_column(label_col, "labels")
    encoded.set_format("torch")
    return encoded


# ========================= 评估 =========================

def compute_metrics(eval_preds, eval_loss: float) -> Dict[str, float]:
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    ppl = math.exp(eval_loss) if eval_loss < 10 else float('inf')
    return {"accuracy": acc, "perplexity": ppl}


def create_peft_model(method: str, num_labels: int, pad_token_id: int):
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels, trust_remote_code=True,
    ).to(DEVICE)
    base.config.pad_token_id = pad_token_id

    if method == "LoRA":
        config = LoraConfig(
            r=LORA_R, lora_alpha=int(LORA_ALPHA), target_modules=TARGET_MODULES,
            lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.SEQ_CLS,
        )
        model = get_peft_model(base, config)

    elif method == "DoRA":
        try:
            config = LoraConfig(
                r=LORA_R, lora_alpha=int(LORA_ALPHA), target_modules=TARGET_MODULES,
                lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.SEQ_CLS,
                use_dora=True,
            )
            model = get_peft_model(base, config)
        except Exception as e:
            print(f"[Warn] DoRA fallback: {e}")
            config = LoraConfig(
                r=LORA_R, lora_alpha=int(LORA_ALPHA), target_modules=TARGET_MODULES,
                lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.SEQ_CLS,
            )
            model = get_peft_model(base, config)
            for module in model.named_modules():
                if hasattr(module, 'lora_A') and hasattr(module, 'base_layer'):
                    module.magnitude = nn.Parameter(module.base_layer.weight.norm(p=2, dim=1).detach().clone())

    elif method == "ReLoRA":
        config = LoraConfig(
            r=LORA_R, lora_alpha=int(LORA_ALPHA), target_modules=TARGET_MODULES,
            lora_dropout=LORA_DROPOUT, bias="none", task_type=TaskType.SEQ_CLS,
        )
        model = get_peft_model(base, config)

    elif method == "Aeloru":
        cfg = AeloruConfig(r=LORA_R, lora_alpha=LORA_ALPHA, LoRA_lr=LR)
        model = inject_aeloru(base, target_names=TARGET_MODULES, cfg=cfg)

    else:
        raise ValueError(f"Unknown method: {method}")

    return model


# ========================= 训练 =========================

def train_and_evaluate(method: str, dataset_cfg: Dict, tokenizer, pad_token_id: int):
    print(f"\n{'='*60}")
    print(f"Method: {method:8s} | Dataset: {dataset_cfg['config'].upper()}")
    print(f"{'='*60}")

    encoded = load_and_preprocess_dataset(dataset_cfg, tokenizer)
    collator = DataCollatorWithPadding(tokenizer)
    train_loader = DataLoader(encoded["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
    val_key = "validation_matched" if dataset_cfg["config"] == "mnli" else "validation"
    eval_loader = DataLoader(encoded[val_key], batch_size=BATCH_SIZE, collate_fn=collator)

    model = create_peft_model(method, dataset_cfg["num_labels"], pad_token_id)
    model.to(DEVICE)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
    )

    relora_trainer = ReLoRATrainer(model, merge_every=1000, lr=LR) if method == "ReLoRA" else None

    aeloru_layers = [m for m in model.modules() if isinstance(m, AeloruLayer)] if method == "Aeloru" else []

    epoch_train_losses = []
    epoch_eval_losses = []
    global_step = 0
    best_eval_acc = 0.0
    final_metrics = {}

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for batch in pbar:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()

            # bf16 前向（无需 GradScaler）
            with autocast(enabled=USE_AMP, dtype=AMP_DTYPE):
                outputs = model(**batch)
                loss = outputs.loss
                if method == "Aeloru" and aeloru_layers:
                    ortho_loss = sum(layer.ortho_penalty() for layer in aeloru_layers if hasattr(layer, 'ortho_penalty'))
                    loss = loss + ortho_loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()

            if method == "Aeloru" and aeloru_layers:
                for layer in aeloru_layers:
                    if hasattr(layer, 'fisher_update') and layer.lora_A.grad is not None:
                        grad_delta = (layer.lora_A.grad @ layer.lora_B.detach().t() +
                                      layer.lora_A.detach() @ layer.lora_B.grad.t()) * layer.scaling
                        layer.fisher_update(grad_delta)

            if relora_trainer:
                relora_trainer.maybe_merge_and_reinit(global_step)

            global_step += 1
            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_train_loss = epoch_loss / len(train_loader)
        epoch_train_losses.append(avg_train_loss)

        # 评估
        model.eval()
        all_logits, all_labels = [], []
        eval_loss_sum = 0.0
        with torch.no_grad():
            for batch in eval_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = model(**batch)
                eval_loss_sum += outputs.loss.item() * batch["input_ids"].size(0)
                all_logits.append(outputs.logits.float().cpu().numpy())  # bf16→fp32
                all_labels.append(batch["labels"].cpu().numpy())

        eval_loss = eval_loss_sum / len(eval_loader.dataset)
        epoch_eval_losses.append(eval_loss)

        metrics = compute_metrics((np.concatenate(all_logits), np.concatenate(all_labels)), eval_loss)
        print(f"  Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
              f"Eval Loss: {eval_loss:.4f} | Acc: {metrics['accuracy']:.4f} | PPL: {metrics['perplexity']:.2f}")

        if metrics["accuracy"] > best_eval_acc:
            best_eval_acc = metrics["accuracy"]
            final_metrics = metrics.copy()

    del model, optimizer
    torch.cuda.empty_cache()
    return final_metrics, epoch_train_losses, epoch_eval_losses


# ========================= 绘图 =========================

def plot_all_loss_curves(all_results: Dict):
    methods = ["LoRA", "DoRA", "ReLoRA", "Aeloru"]
    datasets = [cfg["config"].upper() for cfg in DATASET_CONFIGS]
    fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 4 * len(datasets)))
    if len(datasets) == 1:
        axes = [axes]
    colors = {"LoRA": "#1f77b4", "DoRA": "#ff7f0e", "ReLoRA": "#2ca02c", "Aeloru": "#d62728"}

    for idx, ds_name in enumerate(datasets):
        ax = axes[idx]
        for method in methods:
            if ds_name in all_results.get(method, {}):
                _, train_losses, eval_losses = all_results[method][ds_name]
                epochs = list(range(1, len(train_losses) + 1))
                ax.plot(epochs, train_losses, color=colors[method], linestyle='-', marker='o', label=f"{method} Train", alpha=0.8)
                ax.plot(epochs, eval_losses, color=colors[method], linestyle='--', marker='s', label=f"{method} Eval", alpha=0.8)
        ax.set_title(f"Loss Curves - {ds_name}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Epoch", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.legend(loc='upper right', fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(epochs)

    plt.tight_layout()
    plt.savefig("loss_curves_all_datasets.png", dpi=150, bbox_inches='tight')
    print("\nLoss 曲线图已保存至 loss_curves_all_datasets.png")
    plt.show()


# ========================= 主入口 =========================

def run_experiment():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    print(f"[Tokenizer] pad_token: {tokenizer.pad_token}, pad_token_id: {pad_token_id}")
    print(f"[AMP] dtype={AMP_DTYPE}, enabled={USE_AMP}")

    methods = ["LoRA", "DoRA", "ReLoRA", "Aeloru"]
    results = defaultdict(dict)
    all_loss_records = defaultdict(dict)

    for ds_cfg in DATASET_CONFIGS:
        ds_name = ds_cfg["config"].upper()
        for method in methods:
            try:
                metrics, train_losses, eval_losses = train_and_evaluate(method, ds_cfg, tokenizer, pad_token_id)
                results[method][ds_name] = metrics
                all_loss_records[method][ds_name] = (metrics, train_losses, eval_losses)
            except Exception as e:
                print(f"[ERROR] {method} on {ds_name}: {e}")
                import traceback
                traceback.print_exc()
                results[method][ds_name] = {"accuracy": 0.0, "perplexity": float('inf')}
                all_loss_records[method][ds_name] = (
                    {"accuracy": 0.0, "perplexity": float('inf')},
                    [float('inf')] * NUM_EPOCHS, [float('inf')] * NUM_EPOCHS,
                )

    rows = []
    for method in methods:
        row = {"Method": method}
        accs, ppls = [], []
        for ds_cfg in DATASET_CONFIGS:
            ds_name = ds_cfg["config"].upper()
            m = results[method].get(ds_name, {"accuracy": 0.0, "perplexity": float('inf')})
            row[f"{ds_name}_Acc"] = f"{m['accuracy']:.4f}"
            row[f"{ds_name}_PPL"] = f"{m['perplexity']:.2f}"
            accs.append(m["accuracy"])
            ppls.append(m["perplexity"] if m["perplexity"] != float('inf') else 100)
        row["Avg_Acc"] = f"{np.mean(accs):.4f}"
        row["Avg_PPL"] = f"{np.mean(ppls):.2f}"
        rows.append(row)

    df = pd.DataFrame(rows)
    print("\n" + "="*80)
    print("实验结果汇总表")
    print("="*80)
    print(df.to_string(index=False))
    df.to_csv("peft_comparison_results.csv", index=False, encoding="utf-8-sig")
    print("\n结果已保存至 peft_comparison_results.csv")
    plot_all_loss_curves(all_loss_records)
    return df


if __name__ == "__main__":
    df = run_experiment()

[Tokenizer] pad_token: <|endoftext|>, pad_token_id: 151643
[AMP] dtype=torch.bfloat16, enabled=True

Method: LoRA     | Dataset: SST2


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4998.62it/s]
Qwen2ForSequenceClassification LOAD REPORT from: ../../models/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1/3: 100%|██████████| 4210/4210 [09:35<00:00,  7.31it/s, loss=0.0122]


  Epoch 1 | Train Loss: 0.4559 | Eval Loss: 0.2147 | Acc: 0.9312 | PPL: 1.24


Epoch 2/3: 100%|██████████| 4210/4210 [12:03<00:00,  5.82it/s, loss=0.2237]


  Epoch 2 | Train Loss: 0.1744 | Eval Loss: 0.2210 | Acc: 0.9220 | PPL: 1.25


...

  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru] Injected into q_proj (in=896, out=896, r=8)
  [Aeloru] Injected into v_proj (in=896, out=128, r=8)
  [Aeloru]

Epoch 1/3:   0%|          | 0/230 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "C:\Users\ZhuanZ1\AppData\Local\Temp\ipykernel_18196\3618242479.py", line 297, in run_experiment
    metrics, train_losses, eval_losses = train_and_evaluate(method, ds_cfg, tokenizer, pad_token_id)
                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ZhuanZ1\AppData\Local\Temp\ipykernel_18196\3618242479.py", line 191, in train_and_evaluate
    outputs = model(**batch)
              ^^^^^^^^^^^^^^
  File "e:\Conda\envs\Qelys\Lib\site-packages\torch\nn\modules\module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Conda\envs\Qelys\Lib\site-packages\torch\nn\modules\module.py", line 1750, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Conda\envs\Qelys\Lib\site-packages\transformers\utils

[ERROR] Aeloru on MRPC: outer: Expected 1-D argument self, but got 2-D

Method: LoRA     | Dataset: QNLI


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 8820.13it/s]
Qwen2ForSequenceClassification LOAD REPORT from: ../../models/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1/3:  66%|██████▌   | 4331/6547 [35:26<18:08,  2.04it/s, loss=0.6787]   


KeyboardInterrupt: 

## 连续学习核心对比实验（证明 "核心优势"）
> 实验目的：这是整篇论文最重要的实验，直接证明 Aeloru 在缓解灾难性遗忘上的核心优势。
### 实验设计：
1. 定义连续学习任务序列：Alpaca → GSM8K → SQuAD → IMDB → TriviaQA
2. 按顺序训练 5 个任务，每个任务训练 3 个 epoch，训练完一个任务后，立即在所有之前学过的任务上测试性能
3. 记录每个方法在每个任务学习完成后的性能变化
4. 计算最终的平均准确率和平均遗忘率

## 模块消融实验（证明 "每个创新点都有用"）
>实验目的：证明提出的每一个模块（Hi-DoRA、Hebbian-Fisher、Hong Wen）都对最终性能有贡献
### 实验设计：
   以完整的 Aeloru 为基线，逐个关闭每个模块，得到 5 个变体：
   - Aeloru w/o Hi-DoRA：用普通 LoRA 代替 Hi-DoRA
   - Aeloru w/o Hebbian-Fisher：关闭 Fisher 门控和 Hebbian 更新
   - Aeloru w/o Hong Wen：关闭认知状态机，用固定学习率训练
   - Aeloru w/o 双轨记忆：去掉外置累积缓冲区，只用单个 LoRA
   - 完整 Aeloru
---
在上述连续学习任务序列上，测试所有变体的性能
对比平均遗忘率和平均准确率的变化

## 效率对比实验（证明 "和主流方法一样快"）
> 实验目的：证明 Aeloru 虽然引入了更多机制，但计算开销和主流方法相当，没有明显的速度损失。
### 实验设计：
1. 在相同的硬件和超参数下，记录 4 种方法训练 100 步的平均时间
2. 记录每个方法的峰值显存占用